**Анализ эффективности маркетинговых компаний в социальных сетях для выявления закономерностей вовлечения аудитории**

Комаров Иван ИД24-2

**Бизнес-проблема:**
Высокие показатели охвата, кликов и вовлечённости не всегда означают эффективное использование рекламного бюджета. Поэтому необходимо определить, какие характеристики рекламных кампаний и аудитории связаны с более высокой финансовой отдачей.

**Бизнес-цель:**
Повысить эффективность расходования рекламного бюджета за счёт выявления наиболее результативных маркетинговых стратегий и сегментов аудитории.

**Критерии достижения:**
- выявить стратегии и сегменты аудитории с более высокой финансовой эффективностью;
- определить направления перераспределения рекламного бюджета;
- проверить, позволяет ли учёт дополнительных факторов улучшить прогноз ROAS;
- оценить потенциальное повышение финансовой отдачи на основе ретроспективного анализа.

**Практическая ценность:**
Результаты могут использоваться для выбора более эффективных рекламных стратегий, сегментов аудитории и поддержки решений по распределению рекламного бюджета.

**Объект исследования:**
Онлайн-маркетинговые кампании в социальных сетях.

**Предмет исследования:**
Методы анализа разнородных данных и машинного обучения для оценки финансовой эффективности маркетинговых кампаний и выявления закономерностей вовлечения аудитории.

**Границы исследования:**
Исследование проводится на данных рекламных кампаний Meta Ads в сфере e-commerce. В качестве предметной области рассматриваются товары кофейной тематики. Используются недельные данные рекламных кампаний и внешние источники, характеризующие информационный и потребительский контекст.

**Исследовательский вопрос:**
Какие характеристики рекламных кампаний, аудитории и внешнего контекста связаны с различиями в финансовой эффективности маркетинговых кампаний?

**Гипотеза:**

H1: характеристики рекламных кампаний, аудитории и внешний контекст связаны с ROAS и позволяют улучшить его прогноз.

H0: эти факторы не имеют значимой связи с ROAS и не улучшают прогноз.

**2.2. Подбор и изучение источников данных**

В качестве предметной области выбран сегмент e-commerce, связанный с кофейной продукцией. Выбор обусловлен тем, что основной набор данных содержит реальные рекламные кампании Meta Ads для интернет-магазина данной тематики, а внешние источники позволяют дополнить их информацией об общественном интересе, отраслевом новостном фоне и потребительских обзорах за тот же временной период. Непосредственно для самого исследования были выбраны четыре взаимодополняющих источника. Они относятся к одной предметной области, имеют общий календарный период и могут быть приведены к одной недельной шкале. Mendeley содержит целевой показатель ROAS и характеристики кампаний. Остальные источники формируют наблюдаемый внешний контекст недели.

| Источник | Происхождение и формат | Исходный grain | Фактический объём и период | Роль и способ объединения |
|---|---|---|---|---|
| Mendeley Meta Ads | Semerádová, Weinlich; CSV; DOI `10.17632/hh7xps83z5.1`; CC BY 4.0 | кампания × неделя | 1 872 строки; 36 кампаний; 52 недели; 04.08.2024–27.07.2025 | основной реляционный набор; левая сторона объединения |
| Wikimedia Analytics API | официальный Pageviews REST API; JSON | статья × тип доступа × день | 3 276 записей; 3 статьи × 3 типа доступа × 364 дня; 04.08.2024–02.08.2025 | показатель общественного интереса; агрегация по неделе и `LEFT JOIN` по `Date` |
| Daily Coffee News | публичный RSS издательства Roast Magazine; XML | публикация | 624 публикации; 05.08.2024–01.08.2025 | отраслевой новостной контекст; агрегация по неделе и `LEFT JOIN` по `Date` |
| CoffeeReview | официальный RSS обзоров; XML с неструктурированным текстом | обзор | 604 обзора; 05.08.2024–31.07.2025 | профессиональный обзорный и текстовый контекст; агрегация по неделе и `LEFT JOIN` по `Date` |

Внешние данные собирались до 02.08.2025, чтобы полностью охватить последнюю неделю наблюдений основного набора.

**Общие правила загрузки и проверки**

Следующая ячейка подключает библиотеки и определяет единые функции для загрузки, очистки HTML, формирования воскресной недели и первичного профилирования. Проверенные ответы API и RSS сохраняются в локальном кэше `course_data`: повторный запуск использует те же исходные записи, а при отсутствии кэша обращается к официальным URL. Это снижает нагрузку на источники и делает результат воспроизводимым.

In [1]:
from pathlib import Path
from io import StringIO
import html
import json
import re
import time
import urllib.parse
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import display
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import TfidfVectorizer

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

DATA_DIR = Path("course_data")
DATA_DIR.mkdir(exist_ok=True)
START_DATE = pd.Timestamp("2024-08-04")
END_DATE = pd.Timestamp("2025-08-02")
EXPECTED_WEEKS = pd.date_range(START_DATE, "2025-07-27", freq="7D")

HEADERS = {
    "User-Agent": "academic-coursework/1.0 (public data research; contact: student project)"
}


def sunday_week_start(values):
    """Приводит дату к воскресенью, с которого начинается соответствующая неделя."""
    dates = pd.to_datetime(values, utc=True, errors="raise")
    if isinstance(dates, pd.Series):
        dates = dates.dt.tz_convert(None).dt.normalize()
        dayofweek = dates.dt.dayofweek
    else:
        dates = dates.tz_convert(None).normalize()
        dayofweek = dates.dayofweek
    return dates - pd.to_timedelta((dayofweek + 1) % 7, unit="D")


def html_to_text(value):
    """Удаляет HTML, декодирует сущности и нормализует пробелы."""
    text = BeautifulSoup(str(value or ""), "html.parser").get_text(" ")
    text = html.unescape(text)
    return re.sub(r"\s+", " ", text).strip()


def compact_profile(df, name, date_col=None, key_cols=None):
    print(f"{name}: shape={df.shape}")
    print("Типы данных:")
    display(df.dtypes.rename("dtype").to_frame())
    missing = df.isna().sum()
    print(f"Пропусков всего: {int(missing.sum())}")
    display(missing[missing.gt(0)].rename("missing").to_frame())
    if date_col is not None:
        raw_dates = df[date_col].astype("string")
        date_format = "%Y%m%d%H" if raw_dates.str.fullmatch(r"\d{10}").all() else None
        dates = pd.to_datetime(df[date_col], format=date_format, utc=True, errors="coerce")
        print("Диапазон дат:", dates.min(), "—", dates.max())
    if key_cols is not None:
        print(f"Дубликатов ключа {key_cols}: {int(df.duplicated(key_cols).sum())}")
    display(df.head(3))


def parse_rss_items(xml_bytes):
    root = ET.fromstring(xml_bytes)
    ns = {
        "dc": "http://purl.org/dc/elements/1.1/",
        "content": "http://purl.org/rss/1.0/modules/content/",
    }
    records = []
    for item in root.findall(".//item"):
        def value(tag):
            node = item.find(tag, ns)
            return "" if node is None or node.text is None else node.text
        records.append({
            "title": value("title"),
            "link": value("link"),
            "guid": value("guid"),
            "creator": value("dc:creator"),
            "pubDate": value("pubDate"),
            "description": value("description"),
            "content_encoded": value("content:encoded"),
            "categories": [node.text for node in item.findall("category") if node.text],
        })
    return records


def crawl_rss(feed_template, start_date, end_date, max_pages=250):
    """Читает архивные RSS-страницы до прохождения нижней границы периода."""
    collected = []
    crossed_start = False
    for page in range(1, max_pages + 1):
        url = feed_template.format(page=page)
        response = requests.get(url, headers=HEADERS, timeout=60)
        response.raise_for_status()
        items = parse_rss_items(response.content)
        if not items:
            break
        page_dates = pd.to_datetime([x["pubDate"] for x in items], utc=True, errors="coerce")
        for record, pub_date in zip(items, page_dates):
            if pd.notna(pub_date) and start_date <= pub_date.tz_convert(None) <= end_date + pd.Timedelta(days=1):
                record["source_page"] = page
                collected.append(record)
        valid_dates = page_dates[~pd.isna(page_dates)]
        if len(valid_dates) and valid_dates.min().tz_convert(None) < start_date:
            crossed_start = True
            break
        time.sleep(0.05)
    if not crossed_start:
        raise RuntimeError("RSS-архив не был дочитан до начала исследуемого периода")
    return collected

print("Каталог данных:", DATA_DIR.resolve())
print("Целевой недельный диапазон:", EXPECTED_WEEKS.min().date(), "—", EXPECTED_WEEKS.max().date())

Каталог данных: C:\Users\Ivan Mikhaylov\Downloads\course_data
Целевой недельный диапазон: 2024-08-04 — 2025-07-27


Общие функции задают одинаковое правило недели для всех источников и позволяют далее проверять фактическую структуру каждого набора сопоставимым способом.

**Источник 1. Рекламные кампании Meta Ads из Mendeley Data**

Основной набор опубликован Т. Семерадовой и П. Вайнлих в репозитории Mendeley Data и связан со статьёй о широте и глубине персонализации рекламы. Набор содержит 36 повторно наблюдаемых кампаний: каждая конфигурация представлена в течение 52 недель. Поле `ROAS` используется как опубликованный целевой показатель.

In [2]:
MENDELEY_PATH = DATA_DIR / "mendeley_campaign.csv"
MENDELEY_PAGE = "https://data.mendeley.com/datasets/hh7xps83z5/1"
MENDELEY_FILE_URL = "https://data.mendeley.com/public-files/datasets/hh7xps83z5/files/04b5d7d8-82f6-4e77-a5f2-45ca073fe12d/file_downloaded"

if not MENDELEY_PATH.exists():
    try:
        from playwright.sync_api import sync_playwright
        with sync_playwright() as playwright:
            browser = playwright.chromium.launch(headless=True)
            page = browser.new_page()
            page.goto(MENDELEY_PAGE, wait_until="domcontentloaded", timeout=120_000)
            response = page.request.get(MENDELEY_FILE_URL, timeout=120_000)
            if not response.ok:
                raise RuntimeError(f"Mendeley download failed: HTTP {response.status}")
            MENDELEY_PATH.write_bytes(response.body())
            browser.close()
    except Exception as exc:
        raise RuntimeError(
            "Не удалось получить официальный CSV Mendeley. "
            "Сохраните Coffeeshop_Personalization_Campaign_Data.csv как "
            f"{MENDELEY_PATH}. Причина: {exc}"
        ) from exc

campaign_raw = pd.read_csv(MENDELEY_PATH)
compact_profile(
    campaign_raw,
    "Mendeley Meta Ads",
    date_col="Date",
    key_cols=["Campaign_ID", "Date"],
)
print("Уникальных кампаний:", campaign_raw["Campaign_ID"].nunique())
print("Уникальных недель:", campaign_raw["Date"].nunique())
print("Уровни Funnel:", sorted(campaign_raw["Funnel"].unique()))
print("Уровни Audience:", sorted(campaign_raw["Audience"].unique()))

Mendeley Meta Ads: shape=(1872, 11)
Типы данных:


,dtype
Campaign_ID,str
Date,str
Funnel,str
Retargeting,str
Audience,str
Creative,str
Audience_Size,int64
CTR,float64
Conversions,int64
ROAS,float64


Пропусков всего: 0


,missing


Диапазон дат: 2024-08-04 00:00:00+00:00 — 2025-07-27 00:00:00+00:00
Дубликатов ключа ['Campaign_ID', 'Date']: 0


,Campaign_ID,Date,Funnel,Retargeting,Audience,Creative,Audience_Size,CTR,Conversions,ROAS,Spend
0,CAMP_01,2024-08-04,TOFU,Standard,Custom,Low,902,1.26,0,2.52,197.87
1,CAMP_01,2024-08-11,TOFU,Standard,Custom,Low,921,0.49,0,2.71,202.83
2,CAMP_01,2024-08-18,TOFU,Standard,Custom,Low,1463,1.08,0,2.39,420.92


Уникальных кампаний: 36
Уникальных недель: 52
Уровни Funnel: ['BOFU', 'MOFU', 'TOFU']
Уровни Audience: ['Custom', 'Lookalike_1%', 'Lookalike_5%']


Проверка подтверждает 1 872 строки, 36 кампаний и 52 воскресные даты без пропусков. Каждая дата содержит 36 кампаний, а каждая кампания наблюдается 52 раза.

**Шкалы и характеристики Mendeley:**

| Поле | Шкала |
|---|---|
| `Campaign_ID` | номинальная |
| `Date` | интервальная |
| `Funnel` | порядковая |
| `Retargeting` | дихотомическая |
| `Audience` | категориальная |
| `Creative` | порядковая |
| `Audience_Size` | абсолютная |
| `Conversions` | абсолютная |
| `CTR` | отношений |
| `ROAS` | отношений |
| `Spend` | отношений |

**Источник 2. Wikimedia Analytics API**

Второй источник является официальным программным интерфейсом Wikimedia, который предоставляет статистику просмотров страниц проектов Wikimedia, включая Wikipedia. Через API можно получать временные ряды по отдельным статьям с разбивкой по датам и типам доступа.

Источник выбран как независимый показатель внешнего интереса аудитории к кофейной тематике. Для оценки внешнего интереса выбраны статьи Coffee, Coffeehouse и Espresso, поскольку они отражают соответственно общий интерес к категории кофе, интерес к кофейной культуре и интерес к одному из наиболее распространённых видов кофейных напитков. Просмотры этих статей позволяют оценить, насколько меняется общественный интерес к предметной области во времени, независимо от конкретных рекламных кампаний. Эти данные используются как внешний фактор, потенциально связанный с изменением ROAS.

На уровне недели рассчитываются общий объём просмотров, максимальный дневной интерес, стандартное отклонение дневных просмотров и доля просмотров за выходные. Эти показатели описывают соответственно масштаб, пик, нестабильность и недельную структуру интереса.

In [3]:
WIKI_PATH = DATA_DIR / "wikimedia_pageviews.json"
ARTICLES = ["Coffee", "Espresso", "Coffeehouse"]
ACCESS_TYPES = ["desktop", "mobile-web", "mobile-app"]

if WIKI_PATH.exists():
    wiki_payload = json.loads(WIKI_PATH.read_text(encoding="utf-8"))
else:
    wiki_items = []
    for article in ARTICLES:
        for access in ACCESS_TYPES:
            encoded_article = urllib.parse.quote(article, safe="")
            url = (
                "https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/"
                f"en.wikipedia/{access}/user/{encoded_article}/daily/2024080400/2025080200"
            )
            response = requests.get(url, headers=HEADERS, timeout=60)
            response.raise_for_status()
            wiki_items.extend(response.json()["items"])
    wiki_payload = {"items": wiki_items}
    WIKI_PATH.write_text(json.dumps(wiki_payload, ensure_ascii=False), encoding="utf-8")

wiki_raw = pd.json_normalize(wiki_payload["items"])
compact_profile(
    wiki_raw,
    "Wikimedia Pageviews",
    date_col="timestamp",
    key_cols=["article", "access", "timestamp"],
)
print("Статьи:", sorted(wiki_raw["article"].unique()))
print("Типы доступа:", sorted(wiki_raw["access"].unique()))
print("Уникальных дневных дат:", wiki_raw["timestamp"].nunique())
print("Диапазон views:", int(wiki_raw["views"].min()), "—", int(wiki_raw["views"].max()))

Wikimedia Pageviews: shape=(3276, 7)
Типы данных:


,dtype
project,str
article,str
granularity,str
timestamp,str
access,str
agent,str
views,int64


Пропусков всего: 0


,missing


Диапазон дат: 2024-08-04 00:00:00+00:00 — 2025-08-02 00:00:00+00:00
Дубликатов ключа ['article', 'access', 'timestamp']: 0


,project,article,granularity,timestamp,access,agent,views
0,en.wikipedia,Coffee,daily,2024080400,desktop,user,740
1,en.wikipedia,Coffee,daily,2024080500,desktop,user,915
2,en.wikipedia,Coffee,daily,2024080600,desktop,user,985


Статьи: ['Coffee', 'Coffeehouse', 'Espresso']
Типы доступа: ['desktop', 'mobile-app', 'mobile-web']
Уникальных дневных дат: 364
Диапазон views: 5 — 1617


Wikimedia API предоставил данные по трём тематическим статьям и трём типам доступа за 364 дня. Пропуски в показателе просмотров отсутствуют, дубликаты по сочетанию `article + access + timestamp` не обнаружены.

Для дальнейшего анализа используются как исходные характеристики API, так и производные недельные показатели, отражающие общий объём интереса, максимальную дневную активность, изменчивость просмотров и долю просмотров в выходные дни.

| Поле | Шкала |
|---|---|
| `article` | категориальная |
| `access` | категориальная |
| `day_type` | дихотомическая |
| `timestamp` | интервальная |
| `views` | абсолютная |
| `wiki_views` | абсолютная |
| `wiki_peak_daily_views` | абсолютная |
| `wiki_daily_views_std` | отношений |
| `wiki_weekend_share` | отношений |

**Источник 3. Daily Coffee News RSS**

Третий источник представляет собой отраслевое издание Roast Magazine, посвящённое кофейной индустрии. В публикациях рассматриваются рыночные новости, продукты, розничная торговля, маркетинг, производство и другие события отрасли.

Источник выбран для оценки информационного фона, в котором проходили маркетинговые кампании. Если в определённые недели кофейная тематика активнее присутствует в отраслевых публикациях, это может отражать изменение внешнего контекста и интереса к рынку, потенциально связанного с эффективностью рекламы.

RSS предоставляет дату публикации, автора, тематические категории, заголовок и текстовое описание материала. Это позволяет учитывать как интенсивность новостного потока, так и его структуру. В дальнейшем публикации агрегируются по неделям, совпадающим с недельной детализацией основного набора рекламных кампаний.

In [4]:
NEWS_PATH = DATA_DIR / "daily_coffee_news_items.json"

if NEWS_PATH.exists():
    news_raw = pd.read_json(NEWS_PATH)
else:
    news_records = crawl_rss(
        "https://dailycoffeenews.com/feed/?paged={page}",
        START_DATE,
        END_DATE,
    )
    news_raw = pd.DataFrame(news_records)
    NEWS_PATH.write_text(news_raw.to_json(orient="records", force_ascii=False, date_format="iso"), encoding="utf-8")

news_date_col = "pubDate" if "pubDate" in news_raw.columns else "date"
compact_profile(
    news_raw,
    "Daily Coffee News",
    date_col=news_date_col,
    key_cols=["guid"],
)
news_categories = sorted({category for values in news_raw["categories"] for category in values})
print("Уникальных авторов:", news_raw["creator"].nunique())
print("Уникальных категорий:", len(news_categories))
print("Примеры категорий:", news_categories[:20])
print("Дубликатов link:", int(news_raw.duplicated("link").sum()))

Daily Coffee News: shape=(624, 10)
Типы данных:


,dtype
date,"datetime64[us, UTC]"
title,str
link,str
guid,str
creator,str
description,str
categories,object
page,int64
Date,datetime64[us]
desc_text,str


Пропусков всего: 0


,missing


Диапазон дат: 2024-08-05 13:20:47+00:00 — 2025-08-01 13:13:46+00:00
Дубликатов ключа ['guid']: 0


,date,title,link,guid,creator,description,categories,page,Date,desc_text
0,2025-08-01 13:13:46+00:00,Black Rock Coffee Bar Reportedly Files Confide...,https://dailycoffeenews.com/2025/08/01/black-r...,https://dailycoffeenews.com/?p=189978,Daily Coffee News Staff,"<div><img width=""620"" height=""414"" src=""https:...","[Business Operations, Corporate, Industry, Mar...",76,2025-07-27,After several years of rapid retail expansion ...
1,2025-08-01 12:05:24+00:00,Weekly Coffee News: Cold Brew Manufacturing Sa...,https://dailycoffeenews.com/2025/08/01/weekly-...,https://dailycoffeenews.com/?p=189820,Daily Coffee News Staff,"<div><img width=""620"" height=""434"" src=""https:...","[Columns, Weekly Coffee News]",76,2025-07-27,Welcome to DCN’s Weekly Coffee News! Subscribe...
2,2025-07-31 16:24:24+00:00,Danone Sues Chobani Over ‘Bright & Mellow’ Col...,https://dailycoffeenews.com/2025/07/31/danone-...,https://dailycoffeenews.com/?p=189953,Nick Brown,"<div><img width=""620"" height=""465"" src=""https:...","[Corporate, Craft/Operations, Industry, Macro,...",76,2025-07-27,Two yogurt giants are now locked into a legal ...


Уникальных авторов: 26
Уникальных категорий: 3558
Примеры категорий: ['"Modulating the Flavor Profile of Coffee"', '1% for the Planet', '1105 West Peachtree', '11th Hour Coffee', '2023 Roaster of the Year', '2024 Year In Review', '2025 Coffee Annuals', '2025 Roaster of the Year', '2026 Roaster of the Year', '3D Printing and Additive Manufacturing', '3D printing', '3Temp', '3Temp Hipster', '3Temp Kobra', '3Temp Puls', '3Temp Wall', '3fconcept', '4C', '517 Coffee', '60 Decibels']
Дубликатов link: 0


За исследуемый период получено 624 публикации без повторов. Материалы написаны 26 авторами и относятся к различным тематическим направлениям, среди которых встречаются Industry, Marketing, Retail, Roasting, Market News и Origin. Это позволяет использовать источник не только для оценки общего объёма новостной активности, но и для характеристики структуры отраслевого информационного фона.

Для дальнейшего анализа используются как исходные признаки публикаций, так и недельные показатели, рассчитанные на их основе.

| Поле | Шкала |
|---|---|
| `creator` | категориальная |
| `category` | категориальная |
| `publication_weekday` | категориальная |
| `title`, `description` | контентная |
| `pubDate` | интервальная |
| `news_count` | абсолютная |
| `news_unique_authors` | абсолютная |
| `news_text_chars` | абсолютная |

Недельные показатели отражают интенсивность публикационной активности, разнообразие авторов и общий объём информационного материала. Эти характеристики далее используются как показатели внешнего отраслевого контекста рекламных кампаний.

**Источник 4. CoffeeReview RSS**

Четвертый специализированный источник состоит из обзоров кофе, содержащих тексты профессиональных оценок и описаний продукции. Он выбран для того, чтобы дополнить данные рекламных кампаний показателями потребительского и экспертного интереса к кофейной тематике.

В отличие от Daily Coffee News, который отражает отраслевой информационный фон, CoffeeReview характеризует именно обзорный и оценочный контекст. Это позволяет учитывать, насколько активно в разные недели публиковались материалы о кофейной продукции и какой объём текстового контента формировался вокруг неё.

Для каждой публикации доступны дата, автор, заголовок, краткое описание и расширенный текст. Эти поля используются как неструктурированные данные, которые далее очищаются, преобразуются в текстовые признаки и агрегируются по неделям для объединения с данными рекламных кампаний.

In [5]:
REVIEW_PATH = DATA_DIR / "coffee_review_items.json"

if REVIEW_PATH.exists():
    reviews_raw = pd.read_json(REVIEW_PATH)
else:
    review_records = crawl_rss(
        "https://www.coffeereview.com/review/feed/?paged={page}",
        START_DATE,
        END_DATE,
    )
    reviews_raw = pd.DataFrame(review_records)
    REVIEW_PATH.write_text(reviews_raw.to_json(orient="records", force_ascii=False, date_format="iso"), encoding="utf-8")

compact_profile(
    reviews_raw,
    "CoffeeReview",
    date_col="pubDate",
    key_cols=["guid"],
)
print("Уникальных авторов:", reviews_raw["creator"].nunique())
print("Непустых description:", int(reviews_raw["description"].fillna("").str.len().gt(0).sum()))
print("Непустых content:encoded:", int(reviews_raw["content_encoded"].fillna("").str.len().gt(0).sum()))
print("Дубликатов link:", int(reviews_raw.duplicated("link").sum()))

CoffeeReview: shape=(604, 12)
Типы данных:


,dtype
pubDate,str
title,str
link,str
guid,str
creator,str
description,str
content_encoded,str
description_text,str
content_text,str
source_page,int64


Пропусков всего: 0


,missing


Диапазон дат: 2024-08-05 13:43:59+00:00 — 2025-07-31 15:03:56+00:00
Дубликатов ключа ['guid']: 0


,pubDate,title,link,guid,creator,description,content_encoded,description_text,content_text,source_page,Date,text_chars
0,2025-07-31T15:03:56.000Z,Colombia Geisha Hila San Adolfo,https://www.coffeereview.com/review/colombia-g...,https://www.coffeereview.com/?post_type=review...,Kim Westerman,"<p>Tropical, spice-toned, lavishly herbal. Pas...","<p><span style=""font-weight: 400;"">Produced by...","Tropical, spice-toned, lavishly herbal. Passio...",Produced by Wilder Lasso of Finca Bella Alejan...,74,2025-07-27,768
1,2025-07-25T19:13:54.000Z,Tio Conejo Caturra Colombia,https://www.coffeereview.com/review/tio-conejo...,https://www.coffeereview.com/?post_type=review...,Kim Westerman,"<p>Stone fruit-driven, delicately floral. Whit...","<p><span style=""font-weight: 400;"">Produced at...","Stone fruit-driven, delicately floral. White p...","Produced at Finca Tío Conejo, entirely of the ...",75,2025-07-20,846
2,2025-07-25T19:13:46.000Z,Sebastian Ramirez Pink Bourbon Colombia,https://www.coffeereview.com/review/sebastian-...,https://www.coffeereview.com/?post_type=review...,Kim Westerman,"<p>Tropical-toned, intensely floral. Pineapple...","<p><span style=""font-weight: 400;"">Produced by...","Tropical-toned, intensely floral. Pineapple, h...","Produced by Sebastian Ramirez, entirely of the...",75,2025-07-20,945


Уникальных авторов: 3
Непустых description: 604
Непустых content:encoded: 604
Дубликатов link: 0


Получено 604 обзора без дубликатов по `guid` и `link`. Текстовые поля `description` и `content:encoded` заполнены во всех публикациях. В исследуемом периоде материалы публиковались тремя авторами. Из даты публикации дополнительно выделяются месяц и день недели, что позволяет учитывать сезонность и особенности публикационной активности.

**Шкалы и характеристики CoffeeReview:**

| Поле | Происхождение | Шкала |
|---|---|---|
| `creator` | исходное | категориальная |
| `publication_month` | производное из `pubDate` | категориальная |
| `publication_weekday` | производное из `pubDate` | категориальная |
| `description`, `content:encoded` | исходный текст | контентная |
| `pubDate` | исходное | интервальная |
| `review_count` | число обзоров за неделю | абсолютная |
| `review_unique_authors` | число авторов за неделю | абсолютная |
| `review_text_chars` | суммарный объём текста за неделю | абсолютная |

Таким образом, источник позволяет учитывать не только количество публикуемых обзоров, но и их временную структуру, авторский состав и объём текстового контента. В дальнейшем текст обзоров будет использоваться для формирования дополнительных признаков, характеризующих содержательный контекст недели.

**Сводная пригодность источников**

| Источник | Не менее 3 категориальных характеристик | Не менее 3 числовых характеристик | Временное поле | Связь с итоговым набором |
|---|---|---|---|---|
| Mendeley | Funnel, Retargeting, Audience | Audience_Size, Conversions, ROAS | Date | уже `Campaign_ID × Date` |
| Wikimedia | article, access, day_type | weekly views, peak, std; дополнительно weekend share | timestamp | агрегация до `Date` |
| Daily Coffee News | creator, category, publication_weekday | publications, authors, text chars | pubDate | агрегация до `Date` |
| CoffeeReview | creator, publication_month, publication_weekday | reviews, authors, text chars | pubDate | агрегация до `Date` |

Все четыре источника содержат необходимый набор категориальных, числовых и временных характеристик, но описывают разные стороны исследуемой предметной области.

Основной набор Mendeley характеризует сами рекламные кампании и их финансовую эффективность. Wikimedia отражает изменение общего интереса аудитории к кофейной тематике. Daily Coffee News описывает интенсивность и структуру отраслевого информационного фона, а CoffeeReview дополняет его данными о публикационной активности и текстовом контексте обзоров продукции.

Для объединения источники приводятся к общей недельной временной шкале. При этом сохраняется исходная единица наблюдения основного набора: одна рекламная кампания в одну неделю. Внешние источники не создают новые наблюдения о кампаниях, а дополняют каждую неделю характеристиками внешнего контекста.

**2.3. Подготовка данных к анализу**

Перед объединением источников данные были приведены к единому виду и проверены на пригодность для дальнейшего анализа. Для каждого набора оценивались структура, типы признаков, наличие пропусков и дубликатов, корректность временных полей и соответствие выбранной единице наблюдения.

Поскольку источники имеют разную структуру и детализацию, подготовка выполнялась отдельно для каждого из них. Данные рекламных кампаний сохранялись на уровне "кампания*неделя", а внешние источники предварительно агрегировались до недельного уровня. Текстовые поля новостей и обзоров очищались от HTML-разметки и технических элементов, после чего использовались для формирования количественных и текстовых характеристик.

Объединение выполнялось по недельной дате. При этом контролировалась уникальность ключей и сохранение исходного числа наблюдений основного набора, чтобы внешние данные только дополняли характеристики рекламных кампаний и не создавали искусственного размножения строк.

Исходные данные сохранялись отдельно от подготовленных таблиц, что позволяет при необходимости сопоставить результат преобразования с первоначальными значениями.

**2.3.1. Подготовка основного набора кампаний**

Сначала проверяется точное соответствие опубликованной схеме. Дата приводится к типу `datetime64`, категориальные поля получают тип `category`, числовые поля преобразуются с запретом нераспознаваемых значений. Затем проверяется, что каждая дата является воскресеньем и что составной ключ не повторяется. Эти проверки защищают дальнейшие объединения от скрытого размножения строк.

In [6]:
EXPECTED_CAMPAIGN_COLUMNS = [
    "Campaign_ID", "Date", "Funnel", "Retargeting", "Audience", "Creative",
    "Audience_Size", "CTR", "Conversions", "ROAS", "Spend",
]
assert campaign_raw.columns.tolist() == EXPECTED_CAMPAIGN_COLUMNS

campaign = campaign_raw.copy()
campaign["Date"] = pd.to_datetime(campaign["Date"], errors="raise")
for column in ["Campaign_ID", "Funnel", "Retargeting", "Audience", "Creative"]:
    campaign[column] = campaign[column].astype("category")
for column in ["Audience_Size", "CTR", "Conversions", "ROAS", "Spend"]:
    campaign[column] = pd.to_numeric(campaign[column], errors="raise")

assert campaign.shape == (1872, 11)
assert campaign.isna().sum().sum() == 0
assert campaign.duplicated(["Campaign_ID", "Date"]).sum() == 0
assert campaign["Campaign_ID"].nunique() == 36
assert campaign["Date"].nunique() == 52
assert set(campaign["Date"].dt.dayofweek) == {6}
assert campaign.groupby("Campaign_ID", observed=True)["Date"].nunique().eq(52).all()
assert campaign.groupby("Date", observed=True)["Campaign_ID"].nunique().eq(36).all()

print("Форма:", campaign.shape)
print("Период:", campaign["Date"].min().date(), "—", campaign["Date"].max().date())
print("Пропусков:", int(campaign.isna().sum().sum()))
print("Дубликатов ключа:", int(campaign.duplicated(["Campaign_ID", "Date"]).sum()))
display(campaign.describe(include="all").T)

Форма: (1872, 11)
Период: 2024-08-04 — 2025-07-27
Пропусков: 0
Дубликатов ключа: 0


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Campaign_ID,1872,36,CAMP_01,52,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Date,1872,NaN,NaN,NaN,2025-01-29 12:00:00,2024-08-04 00:00:00,2024-11-01 06:00:00,2025-01-29 12:00:00,2025-04-28 18:00:00,2025-07-27 00:00:00,NaN
Funnel,1872,3,BOFU,624,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Retargeting,1872,2,Dynamic,936,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Audience,1872,3,Custom,624,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Creative,1872,2,High,936,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Audience_Size,1872.0,NaN,NaN,NaN,1158.804487,800.0,990.75,1160.0,1332.0,1499.0,199.809723
CTR,1872.0,NaN,NaN,NaN,1.192821,0.3,0.75,1.13,1.5,2.5,0.560589
Conversions,1872.0,NaN,NaN,NaN,1.194979,0.0,0.0,1.0,2.0,6.0,1.146144
ROAS,1872.0,NaN,NaN,NaN,2.432393,0.8,1.53,2.26,2.99,5.0,1.09794


Основной набор сохранил 1 872 строки и 11 полей. Пропуски и дубликаты ключа отсутствуют; каждая из 36 кампаний имеет 52 недельных наблюдения. Это подтверждает сбалансированную панель и позволяет использовать её как неизменную "левую" основу всех объединений.

**2.3.2. Недельные показатели Wikimedia**

Данные Wikimedia изначально представлены на дневном уровне, тогда как основной набор рекламных кампаний имеет недельную детализацию. Поэтому для последующего объединения необходимо привести показатели внешнего интереса к той же временной шкале.

Сначала просмотры суммируются по всем выбранным статьям и типам доступа за каждый день. Такой подход позволяет получить единый дневной показатель интереса к кофейной тематике, не разделяя его по отдельным страницам и устройствам.

После этого дневные значения агрегируются по неделям, соответствующим датам основного набора. Для каждой недели рассчитываются несколько характеристик:

- `wiki_views` - общий объём просмотров за неделю как показатель общего уровня интереса
- `wiki_peak_daily_views` - максимальное число просмотров за один день отражает наличие кратковременных всплесков интереса
- `wiki_daily_views_std` - стандартное отклонение дневных просмотров показывает, насколько равномерно интерес распределялся внутри недели
- `wiki_weekend_share` - доля просмотров, пришедшихся на выходные позволяет учитывать различия в характере пользовательской активности в рабочие и выходные дни

Недели формируются с воскресенья по субботу, поскольку именно воскресенье используется как недельная дата в основном наборе рекламных кампаний. Это позволяет объединять источники без смещения временных периодов.

In [7]:
wiki = wiki_raw.copy()
wiki["date"] = pd.to_datetime(wiki["timestamp"], format="%Y%m%d%H", utc=True).dt.tz_convert(None).dt.normalize()
wiki["Date"] = sunday_week_start(wiki["date"])
wiki["day_type"] = np.where(wiki["date"].dt.dayofweek >= 5, "weekend", "weekday")
wiki["views"] = pd.to_numeric(wiki["views"], errors="raise")

assert wiki.duplicated(["article", "access", "date"]).sum() == 0
assert set(wiki["article"]) == set(ARTICLES)
assert set(wiki["access"]) == set(ACCESS_TYPES)
assert wiki.groupby(["article", "access"])["date"].nunique().eq(364).all()

daily_wiki = (
    wiki.groupby(["Date", "date"], as_index=False)
        .agg(daily_views=("views", "sum"))
)
daily_wiki["weekend_views"] = np.where(
    daily_wiki["date"].dt.dayofweek >= 5,
    daily_wiki["daily_views"],
    0,
)

wiki_weekly = (
    daily_wiki.groupby("Date", as_index=False)
    .agg(
        wiki_views=("daily_views", "sum"),
        wiki_peak_daily_views=("daily_views", "max"),
        wiki_daily_views_std=("daily_views", "std"),
        weekend_views=("weekend_views", "sum"),
        observed_days=("date", "nunique"),
    )
)
wiki_weekly["wiki_weekend_share"] = wiki_weekly["weekend_views"] / wiki_weekly["wiki_views"]
wiki_weekly = wiki_weekly.drop(columns=["weekend_views"])

assert wiki_weekly.shape == (52, 6)
assert wiki_weekly["Date"].is_unique
assert wiki_weekly["observed_days"].eq(7).all()
assert wiki_weekly.isna().sum().sum() == 0

print("Форма недельной таблицы:", wiki_weekly.shape)
print("Уникальных недель:", wiki_weekly["Date"].nunique())
print("Дней в каждой неделе:", sorted(wiki_weekly["observed_days"].unique()))
display(wiki_weekly.drop(columns="observed_days").describe().T)
display(wiki_weekly.head(3))

Форма недельной таблицы: (52, 6)
Уникальных недель: 52
Дней в каждой неделе: [np.int64(7)]


,count,mean,min,25%,50%,75%,max,std
Date,52,2025-01-29 12:00:00,2024-08-04 00:00:00,2024-11-01 06:00:00,2025-01-29 12:00:00,2025-04-28 18:00:00,2025-07-27 00:00:00,NaN
wiki_views,52.0,26580.538462,19317.0,23548.5,27670.0,29309.5,31762.0,3453.642774
wiki_peak_daily_views,52.0,4112.461538,2895.0,3642.0,4229.5,4516.75,5557.0,577.70898
wiki_daily_views_std,52.0,217.931139,87.925022,162.007931,205.173107,248.965897,670.996203,96.367079
wiki_weekend_share,52.0,0.274447,0.249628,0.268341,0.27447,0.279822,0.297441,0.008608


,Date,wiki_views,wiki_peak_daily_views,wiki_daily_views_std,observed_days,wiki_weekend_share
0,2024-08-04,26661,3965,122.834189,7,0.274633
1,2024-08-11,27344,4087,142.975189,7,0.276185
2,2024-08-18,27857,4136,142.138263,7,0.272750


Получено 52 уникальные недели, каждая содержит ровно семь календарных дней. Недельный ключ не повторяется, пропусков нет. Поле `observed_days` использовалось только для контроля полноты и не войдёт в итоговую витрину; четыре аналитических показателя Wikimedia сохраняются.

**2.3.3. Подготовка Daily Coffee News**

Публикации Daily Coffee News используются как показатель отраслевого информационного фона. Перед объединением с данными рекламных кампаний их необходимо привести к недельной детализации, поскольку основной набор также организован по неделям.

Текст описаний очищается от HTML-разметки, чтобы дальнейшие количественные характеристики отражали содержательный текст публикаций, а не технические элементы страницы. Повторные записи удаляются по идентификаторам публикаций, поскольку одна и та же новость не должна несколько раз влиять на показатели недели.

После этого каждая публикация относится к соответствующей неделе. Для каждой недели рассчитываются:

- `news_count` - количество опубликованных материалов отражает интенсивность новостной активности
- `news_unique_authors` - число уникальных авторов характеризует разнообразие источников публикационной активности внутри издания
- `news_text_chars` - суммарный объём заголовков и очищенных описаний используется как показатель общего объёма информационного материала

Такое преобразование позволяет представить новостной поток в виде набора недельных характеристик и сопоставить его с финансовой эффективностью рекламных кампаний за те же периоды.

In [8]:
news = news_raw.copy()
news["pubDate"] = pd.to_datetime(news[news_date_col], utc=True, errors="raise")
news = news.loc[
    news["pubDate"].dt.tz_convert(None).between(START_DATE, END_DATE + pd.Timedelta(days=1), inclusive="left")
].copy()
news = news.sort_values("pubDate").drop_duplicates("guid").drop_duplicates("link")
news["Date"] = sunday_week_start(news["pubDate"])
news["publication_weekday"] = news["pubDate"].dt.day_name()
news["description_clean"] = news["description"].map(html_to_text)
news["news_document_chars"] = news["title"].fillna("").str.len() + news["description_clean"].str.len()

assert news["guid"].is_unique
assert news["link"].is_unique
assert news["Date"].between(EXPECTED_WEEKS.min(), EXPECTED_WEEKS.max()).all()

news_weekly = (
    news.groupby("Date", as_index=False)
    .agg(
        news_count=("guid", "size"),
        news_unique_authors=("creator", "nunique"),
        news_text_chars=("news_document_chars", "sum"),
    )
)

assert news_weekly.shape == (52, 4)
assert news_weekly["Date"].is_unique
assert news_weekly.isna().sum().sum() == 0

print("Публикаций после очистки и дедупликации:", len(news))
print("Форма недельной таблицы:", news_weekly.shape)
print("Недель с публикациями:", news_weekly["Date"].nunique())
display(news_weekly.describe().T)
display(news[["title", "creator", "pubDate", "publication_weekday", "description_clean"]].head(3))

Публикаций после очистки и дедупликации:

 624
Форма недельной таблицы: (52, 4)
Недель с публикациями: 52


,count,mean,min,25%,50%,75%,max,std
Date,52,2025-01-29 12:00:00,2024-08-04 00:00:00,2024-11-01 06:00:00,2025-01-29 12:00:00,2025-04-28 18:00:00,2025-07-27 00:00:00,NaN
news_count,52.0,12.0,9.0,11.0,12.0,13.0,15.0,1.066422
news_unique_authors,52.0,3.788462,1.0,3.0,4.0,4.0,7.0,1.090719
news_text_chars,52.0,3198.807692,2365.0,3018.5,3184.0,3386.0,3921.0,289.447211


,title,creator,pubDate,publication_weekday,description_clean
623,Fiorenzato Makes Sense with New Grind-By-Weigh...,Howard Bryman,2024-08-05 13:20:47+00:00,Monday,Italian grinder maker Fiorenzato has added two...
622,Two Webinars on the New USDA Organic Rules Com...,Daily Coffee News Staff,2024-08-05 14:16:58+00:00,Monday,Firedancer Coffee Consultants and organics cer...
621,"Made in Transylvania, the Ousia Coffee Grinder",Howard Bryman,2024-08-06 13:57:05+00:00,Tuesday,The owners of Romanian specialty coffee compan...


После фильтрации периода и дедупликации сохранены 624 публикации. Все 52 недели имеют хотя бы одну новостную публикацию, поэтому недельная таблица не требует искусственного заполнения отсутствующих периодов. Очищенный текст используется только для расчёта фактического объёма материалов, а исходный HTML остаётся доступен для проверки.

**2.3.4. Очистка, аннотирование и векторизация CoffeeReview**

Тексты обзоров CoffeeReview используются как неструктурированный источник, характеризующий содержательный контекст кофейной тематики. Чтобы использовать их в дальнейшем анализе, текст необходимо преобразовать в числовую форму.

Сначала для каждого обзора объединяются краткое описание и основной текст. Из полученного текста удаляются HTML-разметка, ссылки и повторяющиеся служебные фразы RSS. Эти технические элементы не несут полезной информации о содержании обзора и могут искажать текстовый анализ.

После очистки применяется TF-IDF - метод векторизации текста, который оценивает важность слов и словосочетаний для конкретного документа относительно всего набора текстов. В результате каждый обзор представляется набором числовых признаков.

Для выделения основных тематических направлений используется NMF - метод разложения неотрицательной матрицы признаков, позволяющий выявлять группы слов, часто встречающихся вместе. На их основе формируются тематические компоненты, которые затем можно интерпретировать как темы обзоров.

Поскольку текстовые признаки в дальнейшем могут использоваться при прогнозировании ROAS, обучение TF-IDF и NMF выполняется только на более ранней части временного периода. Последние недели преобразуются уже обученными моделями, чтобы избежать использования информации из будущего.

In [9]:
def clean_review_text(description, content):
    combined = f"{description or ''} {content or ''}"
    text = html_to_text(combined)
    text = re.sub(
        r"The post\s+.*?\s+appeared first on\s+Coffee Review\s*\.?",
        " ",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(r"\bhttps?://\S+|\bwww\.\S+", " ", text, flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", text).strip()

reviews = reviews_raw.copy()
reviews["pubDate"] = pd.to_datetime(reviews["pubDate"], utc=True, errors="raise")
reviews = reviews.loc[
    reviews["pubDate"].dt.tz_convert(None).between(START_DATE, END_DATE + pd.Timedelta(days=1), inclusive="left")
].copy()
reviews = reviews.sort_values("pubDate").drop_duplicates("guid").drop_duplicates("link")
reviews["Date"] = sunday_week_start(reviews["pubDate"])
reviews["publication_month"] = reviews["pubDate"].dt.month_name()
reviews["publication_weekday"] = reviews["pubDate"].dt.day_name()
reviews["review_text_raw"] = reviews["description"].fillna("") + " " + reviews["content_encoded"].fillna("")
reviews["review_text_clean"] = [
    clean_review_text(description, content)
    for description, content in zip(reviews["description"], reviews["content_encoded"])
]
reviews["review_document_chars"] = reviews["review_text_clean"].str.len()

assert reviews["guid"].is_unique
assert reviews["link"].is_unique
assert reviews["review_text_clean"].str.len().gt(0).all()
assert ~reviews["review_text_clean"].str.contains("appeared first on Coffee Review", case=False, regex=False).any()

unique_weeks = np.array(sorted(campaign["Date"].unique()))
train_weeks = unique_weeks[:42]
test_weeks = unique_weeks[42:]
text_train_mask = reviews["Date"].isin(train_weeks)

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    min_df=3,
    max_df=0.90,
    max_features=1500,
    ngram_range=(1, 2),
)
tfidf_train = vectorizer.fit_transform(reviews.loc[text_train_mask, "review_text_clean"])
tfidf_all = vectorizer.transform(reviews["review_text_clean"])

nmf = NMF(n_components=3, init="nndsvda", random_state=42, max_iter=500)
nmf.fit(tfidf_train)
topic_weights = nmf.transform(tfidf_all)
reviews["text_topic"] = pd.Categorical(np.argmax(topic_weights, axis=1) + 1)

terms = np.array(vectorizer.get_feature_names_out())
topic_terms = []
for topic_number, weights in enumerate(nmf.components_, start=1):
    top = terms[np.argsort(weights)[-10:][::-1]]
    topic_terms.append({"text_topic": topic_number, "top_terms": ", ".join(top)})

print("Документов после очистки:", len(reviews))
print("Train-документов TF-IDF:", int(text_train_mask.sum()))
print("Test-документов TF-IDF:", int((~text_train_mask).sum()))
print("Граница test-периода:", pd.Timestamp(test_weeks.min()).date())
print("Размер словаря:", len(terms))
print("Форма TF-IDF всех документов:", tfidf_all.shape)
display(pd.DataFrame(topic_terms))
display(reviews[["title", "review_text_raw", "review_text_clean", "text_topic"]].head(2))

Документов после очистки: 604
Train-документов TF-IDF: 479
Test-документов TF-IDF: 125
Граница test-периода: 2025-05-25
Размер словаря: 1500
Форма TF-IDF всех документов: (604, 1500)


,text_topic,top_terms
0,1,"cocoa, sweet, fruit, toned, visit information,..."
1,2,"coffees, 256, 256 5282, 5282 information, 608,..."
2,3,"finest quality, roast finest, taiwan strives, ..."


,title,review_text_raw,review_text_clean,text_topic
603,House Blend #8 Espresso,"<p>Evaluated as espresso. Roast-rounded, crisp...","Evaluated as espresso. Roast-rounded, crisply ...",1
602,Colombia Wilton Benitez Yellow Bourbon,"<p>Complex, richly sweet-tart. Strawberry-guav...","Complex, richly sweet-tart. Strawberry-guava, ...",2


После очистки сохраняются две версии текста: исходная в `review_text_raw` и очищенная в `review_text_clean`. Это позволяет использовать подготовленный текст для анализа и при необходимости сверять его с первоначальным содержанием публикации.

TF-IDF и NMF обучаются только на документах первых 42 недель. Документы последних 10 недель используются только для преобразования уже обученными моделями, поэтому информация из будущего не влияет на формирование текстовых признаков.

Для каждой выделенной темы формируется список наиболее значимых слов. По ним оценивается содержательный смысл темы. До такой проверки номер темы используется только как техническая метка.

**2.3.5. Недельная агрегация CoffeeReview**

Обзоры CoffeeReview публиковались не каждую неделю, поэтому сначала рассчитываются показатели только для недель, в которых действительно были публикации. Для каждой такой недели определяется количество обзоров, число уникальных авторов и общий объём очищенного текста.

После этого таблица приводится к полному календарю из 52 недель, совпадающему с временной шкалой рекламных кампаний. Для недель, в которых обзоры отсутствовали, значения `review_count`, `review_unique_authors` и `review_text_chars` принимаются равными нулю.

В данном случае ноль означает не пропуск или неизвестное значение, а фактически наблюдаемое отсутствие публикаций в соответствующую неделю. Такой подход позволяет сохранить все недели основного набора и корректно использовать интенсивность публикационной активности CoffeeReview как характеристику внешнего контекста.

In [10]:
review_weekly_observed = (
    reviews.groupby("Date", as_index=False)
    .agg(
        review_count=("guid", "size"),
        review_unique_authors=("creator", "nunique"),
        review_text_chars=("review_document_chars", "sum"),
    )
)

review_weekly = pd.DataFrame({"Date": EXPECTED_WEEKS}).merge(
    review_weekly_observed,
    on="Date",
    how="left",
    validate="one_to_one",
)
review_columns = ["review_count", "review_unique_authors", "review_text_chars"]
review_weekly[review_columns] = review_weekly[review_columns].fillna(0).astype("int64")

assert review_weekly.shape == (52, 4)
assert review_weekly["Date"].is_unique
assert review_weekly.isna().sum().sum() == 0

print("Форма недельной таблицы:", review_weekly.shape)
print("Недель с публикациями:", int(review_weekly["review_count"].gt(0).sum()))
print("Недель без публикаций:", int(review_weekly["review_count"].eq(0).sum()))
display(review_weekly.describe().T)

Форма недельной таблицы: (52, 4)
Недель с публикациями: 46
Недель без публикаций: 6


,count,mean,min,25%,50%,75%,max,std
Date,52,2025-01-29 12:00:00,2024-08-04 00:00:00,2024-11-01 06:00:00,2025-01-29 12:00:00,2025-04-28 18:00:00,2025-07-27 00:00:00,NaN
review_count,52.0,11.615385,0.0,2.0,10.0,18.0,44.0,10.449043
review_unique_authors,52.0,1.0,0.0,1.0,1.0,1.0,2.0,0.485071
review_text_chars,52.0,7267.730769,0.0,1398.5,6421.0,11102.5,28527.0,6541.349615


Недельная таблица содержит 52 уникальные даты: в 46 неделях есть обзоры, а в шести зафиксирован наблюдаемый нулевой объём публикаций.

**2.3.6. Объединение источников**

Основной набор содержит повторные наблюдения одних и тех же рекламных кампаний в разные недели. Поэтому одного `Campaign_ID` недостаточно для уникальной идентификации строки: одна и та же кампания встречается 52 раза. Поле `Date` также не является уникальным, поскольку в каждую неделю наблюдаются сразу 36 кампаний.

Уникальной записью является только сочетание `Campaign_ID + Date`, то есть конкретная рекламная кампания в конкретную неделю. Поэтому эта пара используется как составной ключ итогового набора данных.

В качестве основы используется таблица Mendeley, поскольку именно она содержит отдельные рекламные кампании и целевой показатель `ROAS`. Данные Wikimedia, Daily Coffee News и CoffeeReview используются как дополнительные характеристики внешнего контекста соответствующей недели.

Объединение выполняется последовательно, при этом основной набор сохраняется полностью: каждая строка по-прежнему соответствует одной рекламной кампании в одну неделю. Внешние источники только добавляют к ней сведения об интересе к кофейной тематике, отраслевой новостной активности и публикации обзоров.

Перед каждым объединением проверяется, что во внешней таблице каждой неделе соответствует только одна строка. Это необходимо, чтобы одна рекламная кампания не дублировалась из-за нескольких записей внешнего источника.

После каждого этапа дополнительно контролируется количество строк и наличие пропусков. Если число наблюдений остаётся неизменным, значит внешние данные были добавлены без нарушения исходной структуры основного набора.

In [11]:
wiki_features = wiki_weekly.drop(columns="observed_days")
external_tables = [
    ("Wikimedia", wiki_features),
    ("Daily Coffee News", news_weekly),
    ("CoffeeReview", review_weekly),
]

final_df = campaign.copy()
merge_audit = []
for source_name, external in external_tables:
    assert external["Date"].is_unique, f"{source_name}: Date должен быть уникальным"
    before_rows = len(final_df)
    before_columns = final_df.shape[1]
    final_df = final_df.merge(external, on="Date", how="left", validate="many_to_one")
    merge_audit.append({
        "source": source_name,
        "rows_before": before_rows,
        "rows_after": len(final_df),
        "columns_before": before_columns,
        "columns_after": final_df.shape[1],
        "missing_after": int(final_df.isna().sum().sum()),
    })
    assert len(final_df) == before_rows

assert final_df.shape == (1872, 21)
assert final_df["Campaign_ID"].nunique() == 36
assert final_df["Date"].nunique() == 52
assert final_df.duplicated(["Campaign_ID", "Date"]).sum() == 0
assert final_df.isna().sum().sum() == 0

FINAL_PATH = DATA_DIR / "final_analytical_dataset.csv"
final_df.to_csv(FINAL_PATH, index=False)

print("Аудит объединений:")
display(pd.DataFrame(merge_audit))
print("Итоговая форма:", final_df.shape)
print("Кампаний:", final_df["Campaign_ID"].nunique())
print("Недель:", final_df["Date"].nunique())
print("Пропусков:", int(final_df.isna().sum().sum()))
print("Дубликатов Campaign_ID + Date:", int(final_df.duplicated(["Campaign_ID", "Date"]).sum()))
print("Сохранено:", FINAL_PATH.resolve())
display(final_df.head(3))

Аудит объединений:


,source,rows_before,rows_after,columns_before,columns_after,missing_after
0,Wikimedia,1872,1872,11,15,0
1,Daily Coffee News,1872,1872,15,18,0
2,CoffeeReview,1872,1872,18,21,0


Итоговая форма: (1872, 21)
Кампаний: 36
Недель: 52
Пропусков: 0
Дубликатов Campaign_ID + Date: 0
Сохранено: /home/agent/how_to_social_marketing_correct/course_data/final_analytical_dataset.csv


,Campaign_ID,Date,Funnel,Retargeting,Audience,Creative,Audience_Size,CTR,Conversions,ROAS,Spend,wiki_views,wiki_peak_daily_views,wiki_daily_views_std,wiki_weekend_share,news_count,news_unique_authors,news_text_chars,review_count,review_unique_authors,review_text_chars
0,CAMP_01,2024-08-04,TOFU,Standard,Custom,Low,902,1.26,0,2.52,197.87,26661,3965,122.834189,0.274633,13,3,3471,17,2,9191
1,CAMP_01,2024-08-11,TOFU,Standard,Custom,Low,921,0.49,0,2.71,202.83,27344,4087,142.975189,0.276185,11,3,2966,14,1,8347
2,CAMP_01,2024-08-18,TOFU,Standard,Custom,Low,1463,1.08,0,2.39,420.92,27857,4136,142.138263,0.272750,12,3,3183,0,0,0


Все этапы объединения сохранили исходные 1 872 наблюдения. Итоговый набор содержит 36 рекламных кампаний за 52 недели и 21 характеристику. Пропуски и дубликаты по сочетанию `Campaign_ID + Date` отсутствуют. Это показывает, что внешние данные были корректно добавлены к рекламным кампаниям и не изменили исходную единицу наблюдения - "одна кампания за одну неделю".

**Структура итогового набора данных:**

| Группа | Поля | Аналитический смысл и шкала |
|---|---|---|
| Идентификатор и время | `Campaign_ID`, `Date` | идентификатор кампании - номинальная шкала; дата - интервальная |
| Характеристики кампании | `Funnel`, `Retargeting`, `Audience`, `Creative` | порядковые, дихотомические и категориальные характеристики рекламной стратегии и аудитории |
| Размер аудитории | `Audience_Size` | абсолютная шкала |
| Результаты кампании | `CTR`, `Conversions`, `ROAS`, `Spend` | показатели эффективности и рекламных расходов; `ROAS` используется как целевая переменная |
| Интерес Wikimedia | `wiki_views`, `wiki_peak_daily_views`, `wiki_daily_views_std`, `wiki_weekend_share` | характеристики общего уровня, пиков и изменчивости интереса к кофейной тематике |
| Новостной контекст | `news_count`, `news_unique_authors`, `news_text_chars` | показатели интенсивности отраслевой информационной активности |
| Обзорный контекст | `review_count`, `review_unique_authors`, `review_text_chars` | показатели активности публикации обзоров и объёма текстового контента |

Показатели `CTR` и `Conversions` становятся известны уже после запуска рекламы, поэтому их использование для предварительного прогноза может привести к использованию информации, которая ещё недоступна в момент принятия решения. Аналогично показатели внешнего контекста текущей недели становятся известны только после её завершения. 

Поэтому при построении модели будут использоваться значения внешних факторов за предыдущие недели. Это позволит проверить, помогают ли уже известные изменения внешнего интереса и информационного фона прогнозировать последующую эффективность рекламных кампаний.